# Phần A — Harris và hai biến thể

### Bài tập 2 — Cài đặt Harris, Harris–Laplace và ANMS

- **Created**: 2026-11-11T07:35:41+07:00
- **Last Updated**: 2026-11-11T08:03:09+07:00
- **Author**: Xem [nhóm và phân công](../README.md#nhom-va-phan-cong).
- **Objective**: Tự cài đặt, kiểm chứng và phân tích phần A theo đề bài.

## References & Consumed Artifacts

- [Đề bài gốc](../pdf/Computer_Vision%2045.pdf).
- [Hướng dẫn và hợp đồng hàm](../docs/shared/HUONG_DAN_THUC_HIEN.md).
- [Quy ước notebook](../agents/rules/NOTEBOOK_HEADER_CONVENTION.md).
- [Quy tắc báo cáo](../agents/rules/RESULTS_REPORTING.md).
- [Ảnh phần này](../images/phan_a/), [kết quả xuất](../experiments/results/phan_a/).
- Source Code: 11 hàm phần A được định nghĩa trực tiếp trong notebook; chưa có kết quả thí nghiệm.

Hai notebook là bản làm việc; trước khi nộp phải ghép thành một notebook theo đề.


## 1. Thư viện và quy ước cài đặt

Các hàm được cài đặt trực tiếp trong notebook. Không gọi hàm góc /
phân vùng dựng sẵn bên trong hàm phải nộp. Đối chiếu thư viện chỉ ở phần được đề cho phép.

Được dùng NumPy, Matplotlib, math, time; heapq cho SRG. Các hàm SciPy/OpenCV có điều kiện
phải đối chiếu bảng thư viện ở README và đề. Cài thư viện không đồng nghĩa được dùng mọi hàm.

Nhóm tự chọn ảnh, giả thuyết, phép đo, tham số và kết luận.

## 2. Môi trường và dữ liệu — nhóm tự điền

Metadata kernel là Python 3; phiên bản trong metadata phản ánh môi trường lúc dựng khung,
không thay thế việc ghi phiên bản thực tế khi chạy bài.

| Thành phần | Phiên bản / thông tin thực tế |
| --- | --- |
| Python / đường dẫn kernel | Chưa ghi |
| NumPy / SciPy / Matplotlib / OpenCV | Chưa ghi |
| Thiết bị, thời điểm chạy, seed ngẫu nhiên | Chưa ghi |

| Ảnh | Nguồn / quyền sử dụng | Kích thước / kiểu / miền cường độ | Phép chỉnh sửa |
| --- | --- | --- | --- |
| Chưa chọn | Chưa điền | Chưa điền | Chưa điền |

- Mở notebook với working directory là thư mục chứa notebook.
- Đọc ảnh từ `../images/phan_a/`; xuất bảng/hình vào `../experiments/results/phan_a/`.
- Ghi rõ ảnh xám `(H, W)`, miền cường độ và ngưỡng cùng đơn vị.
- Điểm theo `(row, col)`; khi vẽ trục ngang là col, trục dọc là row.
- Quy ước biên, seeds, biến đổi và ghép/đếm: nhóm tự ghi trước khi đo.

Ô dưới khai báo NumPy, Gaussian blur và các hàm hỗ trợ. Ảnh thực nghiệm và thông tin môi trường sẽ được bổ sung khi chạy thí nghiệm.

Quy ước: ảnh float64 giữ nguyên miền cường độ; biên phản xạ; đạo hàm sai phân trung tâm. NMS giữ điểm hòa. Thứ tự hòa: response giảm dần, rồi row/col tăng dần. Harris–Laplace loại tầng đầu/cuối. Thiếu ứng viên thì trả các điểm hiện có. Repeatability ghép lân cận cho từng điểm A, cho phép dùng chung B; mẫu số là số điểm A chiếu hợp lệ, rỗng trả 0.


In [1]:
import numpy as np
from scipy.ndimage import gaussian_filter


def _gray_float(img):
    """Ảnh xám hữu hạn, không tự thay đổi miền cường độ."""
    image = np.asarray(img, dtype=np.float64)
    if image.ndim != 2 or image.size == 0 or not np.isfinite(image).all():
        raise ValueError("img phải là ảnh xám 2D hữu hạn, không rỗng")
    return image


def _positive(value, name):
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"{name} phải dương và hữu hạn")


def _points(points):
    points = np.asarray(points, dtype=np.float64)
    if points.ndim != 2 or points.shape[1] != 3 or not np.isfinite(points).all():
        raise ValueError("points phải có shape (N, 3), giá trị hữu hạn")
    return points


def _count(value):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 0:
        raise ValueError("số lượng phải là số nguyên không âm")
    return int(value)


def _response_order(points):
    return np.lexsort((points[:, 1], points[:, 0], -points[:, 2]))


def _central_gradients(image):
    padded = np.pad(image, 1, mode="symmetric")
    dx = (padded[1:-1, 2:] - padded[1:-1, :-2]) / 2.0
    dy = (padded[2:, 1:-1] - padded[:-2, 1:-1]) / 2.0
    return dx, dy


## 3. Harris cổ điển — tự cài đặt

Các hàm được định nghĩa trong các ô code dưới đây.

### `gaussian_blur(img, sigma)`

**Hợp đồng đầu ra:** Ảnh làm mờ cùng kích thước; gaussian_filter chỉ dùng cho làm mờ Gauss.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [2]:
def gaussian_blur(img, sigma):
    """Làm mờ Gauss, giữ kích thước và miền cường độ ảnh."""
    image = _gray_float(img)
    _positive(sigma, "sigma")
    return gaussian_filter(image, sigma=float(sigma), mode="reflect")


### `structure_tensor(img, sigma_i, sigma_d)`

**Hợp đồng đầu ra:** (Mxx, Mxy, Myy), mỗi mảng cùng kích thước ảnh, đã nhân sigma_d**2.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [3]:
def structure_tensor(img, sigma_i, sigma_d):
    """Tensor tích phân từ đạo hàm sai phân trung tâm, chuẩn hóa sigma_d²."""
    _positive(sigma_i, "sigma_i")
    dx, dy = _central_gradients(gaussian_blur(img, sigma_d))
    factor = float(sigma_d) ** 2
    return tuple(factor * gaussian_blur(term, sigma_i)
                 for term in (dx * dx, dx * dy, dy * dy))


### `harris_response(img, sigma_i, sigma_d, k=0.04)`

**Hợp đồng đầu ra:** Mảng R kiểu float, cùng kích thước ảnh.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [4]:
def harris_response(img, sigma_i, sigma_d, k=0.04):
    """R = det(M) - k * trace(M)²."""
    if not np.isfinite(k) or k <= 0:
        raise ValueError("k phải dương và hữu hạn")
    mxx, mxy, myy = structure_tensor(img, sigma_i, sigma_d)
    return mxx * myy - mxy ** 2 - k * (mxx + myy) ** 2


### `nms_2d(response, window=3)`

**Hợp đồng đầu ra:** Mảng bool đánh dấu cực đại địa phương.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [5]:
def nms_2d(response, window=3):
    """Mặt nạ cực đại; giữ các điểm hòa, bỏ qua vùng ngoài ảnh."""
    response = _gray_float(response)
    if _count(window) < 1 or window % 2 == 0:
        raise ValueError("window phải là số nguyên lẻ dương")
    radius = window // 2
    padded = np.pad(response, radius, mode="constant", constant_values=-np.inf)
    maxima = np.full(response.shape, -np.inf)
    height, width = response.shape
    for row in range(window):
        for col in range(window):
            np.maximum(maxima, padded[row:row + height, col:col + width], out=maxima)
    return response == maxima


### `harris_classic(img, sigma_i=2.0, sigma_d=1.4, k=0.04, thresh_rel=0.01)`

**Hợp đồng đầu ra:** Mảng (N, 3): (row, col, response), giảm dần theo response.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [6]:
def harris_classic(img, sigma_i=2.0, sigma_d=1.4, k=0.04, thresh_rel=0.01):
    """Trả (row, col, response), thứ tự response giảm dần."""
    if not np.isfinite(thresh_rel) or not 0 <= thresh_rel <= 1:
        raise ValueError("thresh_rel phải nằm trong [0, 1]")
    response = harris_response(img, sigma_i, sigma_d, k)
    mask = nms_2d(response) & (response > 0) & (response > thresh_rel * response.max())
    row, col = np.nonzero(mask)
    points = np.column_stack((row, col, response[row, col]))
    return points[_response_order(points)]


## 4. Harris–Laplace — tự cài đặt

Các hàm được định nghĩa trong các ô code dưới đây.

### `scale_pyramid(sigma0=1.2, xi=1.2, n_levels=10)`

**Hợp đồng đầu ra:** Mảng sigma một chiều; ghi rõ số tầng.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [7]:
def scale_pyramid(sigma0=1.2, xi=1.2, n_levels=10):
    """Đúng n_levels tầng: sigma0 * xi**n, n bắt đầu từ 0."""
    _positive(sigma0, "sigma0")
    if not np.isfinite(xi) or xi <= 1:
        raise ValueError("xi phải hữu hạn và lớn hơn 1")
    levels = _count(n_levels)
    with np.errstate(over="ignore"):
        sigmas = float(sigma0) * float(xi) ** np.arange(levels, dtype=float)
    if not np.isfinite(sigmas).all():
        raise ValueError("tháp sigma vượt miền số hữu hạn")
    return sigmas


### `log_normalized(img, sigma)`

**Hợp đồng đầu ra:** LoG chuẩn hóa theo tỉ lệ như hợp đồng trong đề.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [8]:
def log_normalized(img, sigma):
    """sigma² * abs(Lxx + Lyy), đạo hàm bậc hai sai phân trên ảnh Gauss."""
    blurred = gaussian_blur(img, sigma)
    padded = np.pad(blurred, 1, mode="symmetric")
    laplacian = (padded[1:-1, 2:] + padded[1:-1, :-2]
                 + padded[2:, 1:-1] + padded[:-2, 1:-1] - 4 * blurred)
    return float(sigma) ** 2 * np.abs(laplacian)


### `harris_laplace(img, sigmas, k=0.04, thresh_rel=0.01, log_thresh=0.0)`

**Hợp đồng đầu ra:** Mảng (N, 4): (row, col, sigma, response).

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [9]:
def harris_laplace(img, sigmas, k=0.04, thresh_rel=0.01, log_thresh=0.0):
    """Harris không gian + cực đại LoG nghiêm ngặt theo tỉ lệ, cùng tọa độ."""
    image = _gray_float(img)
    sigmas = np.asarray(sigmas, dtype=float)
    if (sigmas.ndim != 1 or not np.isfinite(sigmas).all()
            or np.any(sigmas <= 0) or np.any(np.diff(sigmas) <= 0)):
        raise ValueError("sigmas phải là mảng 1D dương, hữu hạn, tăng nghiêm ngặt")
    if not np.isfinite(log_thresh) or log_thresh < 0:
        raise ValueError("log_thresh phải không âm và hữu hạn")
    if not np.isfinite(thresh_rel) or not 0 <= thresh_rel <= 1:
        raise ValueError("thresh_rel phải nằm trong [0, 1]")
    if not np.isfinite(k) or k <= 0:
        raise ValueError("k phải dương và hữu hạn")
    if len(sigmas) < 3:
        return np.empty((0, 4), dtype=float)
    logs = [log_normalized(image, sigma) for sigma in sigmas]
    selected = []
    for level in range(1, len(sigmas) - 1):
        sigma = sigmas[level]
        candidates = harris_classic(image, sigma, 0.7 * sigma, k, thresh_rel)
        row, col = candidates[:, :2].astype(int).T
        values = logs[level][row, col]
        keep = ((values > log_thresh) & (values > logs[level - 1][row, col])
                & (values > logs[level + 1][row, col]))
        selected.append(np.column_stack((candidates[keep, :2],
                                        np.full(keep.sum(), sigma), candidates[keep, 2])))
    points = np.concatenate(selected, axis=0)
    order = np.lexsort((points[:, 2], points[:, 1], points[:, 0], -points[:, 3]))
    return points[order]


## 5. ANMS và đường cơ sở — tự cài đặt

Các hàm được định nghĩa trong các ô code dưới đây.

### `anms(points, n_keep, c_robust=0.9)`

**Hợp đồng đầu ra:** Mảng (n_keep, 3), giảm dần theo bán kính; ứng viên Harris một tỉ lệ (N, 3).

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [10]:
def anms(points, n_keep, c_robust=0.9):
    """Chọn theo bán kính đến điểm mạnh hơn; O(N²) thời gian, O(N) bộ nhớ."""
    points = _points(points)
    n_keep = _count(n_keep)
    if not np.isfinite(c_robust) or not 0 < c_robust <= 1:
        raise ValueError("c_robust phải thuộc (0, 1]")
    if n_keep == 0:
        return np.empty((0, 3), dtype=float)
    radii_sq = np.full(len(points), np.inf)
    for index, point in enumerate(points):
        stronger = points[:, 2] * c_robust > point[2]
        stronger[index] = False
        if stronger.any():
            delta = points[stronger, :2] - point[:2]
            radii_sq[index] = np.min(np.sum(delta * delta, axis=1))
    order = np.lexsort((points[:, 1], points[:, 0], -points[:, 2], -radii_sq))
    return points[order[:n_keep]].copy()


### `top_n(points, n_keep)`

**Hợp đồng đầu ra:** Mảng (n_keep, 3) chứa các điểm mạnh nhất từ cùng tập ứng viên.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [11]:
def top_n(points, n_keep):
    """Chọn tối đa n_keep điểm mạnh nhất, hòa thì theo row/col."""
    points = _points(points)
    n_keep = _count(n_keep)
    return points[_response_order(points)[:n_keep]].copy()


## 6. Phép đo — tự cài đặt

Các hàm được định nghĩa trong các ô code dưới đây.

### `repeatability(pts_a, pts_b, transform, img_shape, eps=3.0)`

**Hợp đồng đầu ra:** Giá trị trong [0, 1]; transform ánh xạ (row, col) A sang B; chỉ tính điểm chiếu hợp lệ trong B.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; kiểm chứng thí nghiệm nằm ở phần 7.

In [12]:
def repeatability(pts_a, pts_b, transform, img_shape, eps=3.0):
    """Tỷ lệ điểm A chiếu hợp lệ có ít nhất một điểm B cách <= eps.

    transform: callable nhận/trả (N, 2) theo (row, col).
    img_shape: (height, width) ảnh đích. Ghép lân cận cho từng A,
    có thể dùng chung một điểm B; điểm ngoài ảnh không tham gia.
    """
    def coordinates(points):
        array = np.asarray(points, dtype=float)
        if array.ndim != 2 or array.shape[1] < 2 or not np.isfinite(array).all():
            raise ValueError("điểm phải là mảng (N, >=2) hữu hạn")
        return array[:, :2]

    a, b = coordinates(pts_a), coordinates(pts_b)
    shape = np.asarray(img_shape)
    if shape.shape != (2,) or not np.isfinite(shape).all() or np.any(shape <= 0):
        raise ValueError("img_shape phải là (height, width) dương")
    if not np.isfinite(eps) or eps < 0:
        raise ValueError("eps phải không âm và hữu hạn")
    if not callable(transform):
        raise ValueError("transform phải là callable nhận/trả tọa độ (N, 2)")
    if len(a) == 0:
        return 0.0
    projected = np.asarray(transform(a.copy()), dtype=float)
    if projected.shape != a.shape:
        raise ValueError("transform phải trả cùng shape (N, 2)")

    def inside(points):
        return np.isfinite(points).all(axis=1) & ((points >= 0) & (points < shape)).all(axis=1)

    projected, b = projected[inside(projected)], b[inside(b)]
    if len(projected) == 0 or len(b) == 0:
        return 0.0
    matched = sum(np.any(np.sum((b - point) ** 2, axis=1) <= eps ** 2)
                  for point in projected)
    return float(matched / len(projected))


## 7. Kiểm chứng phần A

Hoàn thành đủ năm kiểm chứng với hình và số đo thực. Hàm `grid_occupancy` được đề nói là cho sẵn nhưng chưa có mã/chữ ký trong tài liệu hiện có; cần lấy từ giảng viên, không tự khẳng định đã có định nghĩa chuẩn.

### 7.1. Bàn cờ tổng hợp

Ghi giao điểm chuẩn; đối chiếu sai số vị trí theo yêu cầu ≤ 1 px.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Ảnh / cấu hình | Số điểm | Sai số (px) | Đạt / chưa đạt |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 7.2. Quay 30°

Ghi phép ánh xạ, miền hợp lệ, eps và quy tắc ghép.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Cấu hình | Số điểm trước / sau | Điểm hợp lệ | Repeatability |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 7.3. Phóng 3×

So Harris và Harris–Laplace trên cùng cặp ảnh, phép ghép và eps.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Phương pháp | Số điểm / điểm hợp lệ | Repeatability | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 7.4. Đối chiếu OpenCV

Chỉ dùng cv2.cornerHarris ở phần đối chiếu được phép; so vị trí trong bán kính 2–3 px, không yêu cầu trùng tuyệt đối.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Tham số hai bên | Bán kính (px) | Mức khớp / định nghĩa | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 7.5. ANMS và top_n

Cùng ứng viên Harris một tỉ lệ, cùng N và cùng lưới; ghi nguồn/định nghĩa grid_occupancy.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Phương pháp | N | Lưới / nguồn hàm | Độ phủ |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

## 8. Thí nghiệm A1 — Harris–Laplace so với Harris

Giữ cùng cặp ảnh biến đổi, phép ghép và eps; báo số điểm hợp lệ cho cả hai phương pháp.

### Phiếu thiết kế — nhóm quyết định trước khi chạy

| Trường | Nội dung do nhóm tự điền |
| --- | --- |
| Câu hỏi và giả thuyết | Chưa điền |
| Ảnh, nguồn, lý do chọn và chỉnh sửa | Chưa chọn |
| Biến độc lập và tập giá trị quét | Chưa chọn |
| Yếu tố giữ cố định | Chưa điền |
| Phép đo, định nghĩa, đơn vị và mẫu số | Chưa chọn |
| Seed ngẫu nhiên, số lần lặp, cách tổng hợp | Chưa chọn |
| Bằng chứng cần lưu: ảnh, bảng, đường cong | Chưa điền |

### Kết quả — chỉ điền số đo thật

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.


| Lần chạy | Giá trị tham số quét | Cấu hình cố định | Phương pháp | Đại lượng / đơn vị | Giá trị | Artifact |
| --- | --- | --- | --- | --- | --- | --- |
| Chưa chạy | Chưa chọn | Chưa ghi | Chưa ghi | Chưa chọn | Chưa đo | Chưa có |

Ô dưới để bổ sung khung chạy / vẽ sau khi nhóm chốt thiết kế.


### Phân tích và giới hạn — nhóm tự viết

- Bằng chứng khác biệt giữa hai phương pháp: chưa có.
- Đường cong / điểm chuyển hành vi khi quét tham số: chưa có.
- Đại lượng nào thay đổi, tại bước nào, vì sao gây ra hiện tượng: chưa phân tích.
- Kết quả có khớp giả thuyết không, phạm vi áp dụng và giới hạn: chưa kết luận.

- [ ] Có ít nhất một phép đo và bảng số gốc.
- [ ] Có đường cong quét tham số, tên trục, đơn vị và chú giải.
- [ ] Giải thích cơ chế thay vì chỉ nói phương pháp nào tốt hơn.


## 9. Thí nghiệm A2 — ANMS so với top_n

Dùng cùng tập ứng viên Harris một tỉ lệ, cùng số điểm giữ lại và cùng định nghĩa lưới độ phủ; A2 độc lập với Harris–Laplace.

### Phiếu thiết kế — nhóm quyết định trước khi chạy

| Trường | Nội dung do nhóm tự điền |
| --- | --- |
| Câu hỏi và giả thuyết | Chưa điền |
| Ảnh, nguồn, lý do chọn và chỉnh sửa | Chưa chọn |
| Biến độc lập và tập giá trị quét | Chưa chọn |
| Yếu tố giữ cố định | Chưa điền |
| Phép đo, định nghĩa, đơn vị và mẫu số | Chưa chọn |
| Seed ngẫu nhiên, số lần lặp, cách tổng hợp | Chưa chọn |
| Bằng chứng cần lưu: ảnh, bảng, đường cong | Chưa điền |

### Kết quả — chỉ điền số đo thật

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.


| Lần chạy | Giá trị tham số quét | Cấu hình cố định | Phương pháp | Đại lượng / đơn vị | Giá trị | Artifact |
| --- | --- | --- | --- | --- | --- | --- |
| Chưa chạy | Chưa chọn | Chưa ghi | Chưa ghi | Chưa chọn | Chưa đo | Chưa có |

Ô dưới để bổ sung khung chạy / vẽ sau khi nhóm chốt thiết kế.


### Phân tích và giới hạn — nhóm tự viết

- Bằng chứng khác biệt giữa hai phương pháp: chưa có.
- Đường cong / điểm chuyển hành vi khi quét tham số: chưa có.
- Đại lượng nào thay đổi, tại bước nào, vì sao gây ra hiện tượng: chưa phân tích.
- Kết quả có khớp giả thuyết không, phạm vi áp dụng và giới hạn: chưa kết luận.

- [ ] Có ít nhất một phép đo và bảng số gốc.
- [ ] Có đường cong quét tham số, tên trục, đơn vị và chú giải.
- [ ] Giải thích cơ chế thay vì chỉ nói phương pháp nào tốt hơn.


## Tổng kết và checklist phần A

**Kết luận / hạn chế / công việc còn lại:** Chưa thực hiện.

- [ ] Đủ 11 hàm A đúng chữ ký và hợp đồng đầu ra.
- [ ] Đủ năm kiểm chứng A có hình và số đo.
- [ ] Đủ hai cặp thí nghiệm A1 và A2.

- [ ] Điền nguồn ảnh và tài liệu tham khảo thực tế đã dùng.
- [ ] Restart & Run All với đúng kernel, lưu đủ output thực tế.
- [ ] Trước khi nộp: ghép A và B thành một notebook, chỉnh đường dẫn ảnh theo vị trí mới.

Notebook rỗng chạy được không có nghĩa thuật toán đã qua kiểm chứng.
Đề hiện ghi tên bài nộp `bt1_<Nhom>.ipynb`; xác nhận với giảng viên trước khi đổi tiền tố.
